# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring a FAIR\^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and leverages the open `mlcroissant` standard.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}")


## 2. Data Overview
Review available record sets, their corresponding `@id` values, fields, and fields' `@id`s. This will help determine what data is available for extraction and analysis.

In [ ]:
# List available record sets by their @id and names
print("Record sets available in this dataset:")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- @id: {rs['@id']} | name: {rs.get('name', 'N/A')}")
    # List their fields as well
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for field in fields:
        field_id = field['@id'] if isinstance(field, dict) else field
        print(f"    - field @id: {field_id}")
    print()


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Reference record sets and fields using their `@id`.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records from {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} rows. Columns: {df.columns.tolist()}")
    print()

# If only one record set, select it by default
if len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]
    print(f"Example data preview from '{main_record_set_id}':")
    print(dataframes[main_record_set_id].head())


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filter numeric records, normalize, and group categorical fields, referencing by `@id` only.

In [ ]:
# --- Adjust these variables to match available field @ids in your record set ---
# Example: Using first record set and finding an appropriate numeric and grouping field
df = dataframes[main_record_set_id].copy()

# Try to automatically find a numeric field (e.g., one containing 'abundance', 'percent', 'count', or 'weight')
import numpy as np
numeric_field = None
for c in df.columns:
    if any(x in c.lower() for x in ['abundance', 'percent', 'count', 'weight', 'amount', 'score']):
        try:
            # If the field can be converted to numeric, use it
            df[c] = pd.to_numeric(df[c], errors='coerce')
            if df[c].notnull().sum() > 0:
                numeric_field = c
                break
        except Exception:
            continue

if numeric_field is None:
    numeric_field = df.select_dtypes(include=[np.number]).columns[0]

print(f"Numeric field chosen for analysis: {numeric_field}")

# Pick a group-by field (e.g., one containing 'description', 'sample', 'class', or 'type', but not same as numeric_field)
group_field = None
for c in df.columns:
    if any(x in c.lower() for x in ['description', 'class', 'type', 'sample']) and c != numeric_field:
        group_field = c
        break
if group_field:
    print(f"Group field: {group_field}")

# Filtering records based on a threshold on the numeric field
threshold = df[numeric_field].mean() if np.isfinite(df[numeric_field].mean()) else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered {len(filtered_df)} records with '{numeric_field}' > {threshold:.2f}:")
print(filtered_df[[numeric_field]].head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized '{numeric_field}' for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped data by '{group_field}' (mean of '{numeric_field}'):")
    print(grouped_df.head())


## 5. Visualization
Visualize numeric data distributions and relationships between attributes identified above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the main numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
plt.title(f"Distribution of '{numeric_field}'")
plt.xlabel(numeric_field)
plt.show()

# If group_field is available, plot top groups
if group_field:
    top_n = 10
    group_means = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).head(top_n)
    plt.figure(figsize=(8,4))
    sns.barplot(x=group_means.values, y=group_means.index, orient='h')
    plt.title(f"Top {top_n} {group_field} by mean '{numeric_field}'")
    plt.xlabel(f"Mean {numeric_field}")
    plt.ylabel(group_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR\^2 dataset using its Croissant schema, explored available record sets and fields, performed basic EDA (including filtering, normalization, and grouping), and visualized the protein data. For further analysis, see the detailed documentation or the data dictionary section provided in the Croissant schema metadata.